<a href="https://colab.research.google.com/github/Mohammed-Taher6705/jigsaw-puzzle-matching/blob/main/Version2_Matching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!git clone https://github.com/Mohammed-Taher6705/jigsaw-puzzle-matching.git

fatal: destination path 'jigsaw-puzzle-matching' already exists and is not an empty directory.


In [ ]:
import zipfile
import os

zip_path = "/content/jigsaw-puzzle-matching/Dataset.zip"
extract_path = "/content/Dataset"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Extracted to:", extract_path)
print("Folders inside extracted dataset:", os.listdir(extract_path))


Extracted to: /content/Dataset
Folders inside extracted dataset: ['Dataset']


In [ ]:
import zipfile
import os

zip_path = "/content/jigsaw-puzzle-matching/Task3_output.zip"
extract_path = "/content/Task3_output"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Extracted to:", extract_path)
print("Folders inside extracted dataset:", os.listdir(extract_path))


In [6]:
import os
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim
import time
from itertools import permutations

class CompletePuzzleSolver:
    def __init__(self, dataset_path, correct_path, output_path, ssim_threshold=0.6):
        self.dataset_path = dataset_path
        self.correct_path = correct_path
        self.output_path = output_path
        self.ssim_threshold = ssim_threshold

        # Create output directory structure
        for puzzle_type in ['puzzle_2x2', 'puzzle_4x4', 'puzzle_8x8']:
            os.makedirs(os.path.join(output_path, puzzle_type), exist_ok=True)

        # Store results
        self.results = {}
        self.stats = {
            'total_puzzles': 0,
            'solved_puzzles': 0,
            'algorithm_usage': {},
            'ssim_scores': []
        }

    def load_puzzle_pieces(self, puzzle_type, puzzle_id):
        """Load all pieces for a specific puzzle ID and type"""
        puzzle_folder = os.path.join(self.dataset_path, puzzle_type)
        pieces = {}

        if not os.path.exists(puzzle_folder):
            return pieces

        for filename in os.listdir(puzzle_folder):
            # Try multiple filename patterns
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                try:
                    # Remove extension
                    base_name = os.path.splitext(filename)[0]
                    parts = base_name.split('_')

                    # Debug: print what we're parsing
                    # print(f"Parsing: {filename} -> parts: {parts}")

                    # Pattern 1: {id}_r{row}_c{col} (3 parts) - e.g., "0_r0_c0"
                    if len(parts) == 3:
                        # Check if the first part matches puzzle_id
                        if parts[0] == str(puzzle_id):
                            # Extract row and col from rX and cY
                            row = int(parts[1][1:])  # Remove 'r' from 'r0'
                            col = int(parts[2][1:])  # Remove 'c' from 'c0'

                            img_path = os.path.join(puzzle_folder, filename)
                            img = cv2.imread(img_path)
                            if img is not None:
                                pieces[(row, col)] = img
                                # print(f"Loaded: {filename} -> ({row}, {col})")

                    # Pattern 2: {id}_something_r{row}_c{col} (4 parts)
                    elif len(parts) == 4:
                        # Check if the first part matches puzzle_id
                        if parts[0] == str(puzzle_id):
                            # Extract row and col from rX and cY
                            row_part = parts[2]
                            col_part = parts[3]
                            if row_part.startswith('r') and col_part.startswith('c'):
                                row = int(row_part[1:])
                                col = int(col_part[1:])

                                img_path = os.path.join(puzzle_folder, filename)
                                img = cv2.imread(img_path)
                                if img is not None:
                                    pieces[(row, col)] = img
                                    # print(f"Loaded: {filename} -> ({row}, {col})")

                except Exception as e:
                    # print(f"Error parsing {filename}: {e}")
                    continue

        # Debug: show what we loaded
        # print(f"Loaded {len(pieces)} pieces for puzzle {puzzle_id}")
        return pieces

    def load_correct_image(self, puzzle_id):
        """Load correct image for a puzzle ID"""
        # Try different extensions
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
            correct_file = os.path.join(self.correct_path, f"{puzzle_id}{ext}")
            if os.path.exists(correct_file):
                img = cv2.imread(correct_file)
                if img is not None:
                    return img

        # Check if correct folder has subfolder 'correct'
        if os.path.exists(os.path.join(self.correct_path, 'correct')):
            for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
                correct_file = os.path.join(self.correct_path, 'correct', f"{puzzle_id}{ext}")
                if os.path.exists(correct_file):
                    img = cv2.imread(correct_file)
                    if img is not None:
                        return img

        # Try with 'correct_' prefix
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
            correct_file = os.path.join(self.correct_path, f"correct_{puzzle_id}{ext}")
            if os.path.exists(correct_file):
                img = cv2.imread(correct_file)
                if img is not None:
                    return img

        return None

    def calculate_ssim(self, img1, img2):
        """Calculate SSIM between two images"""
        if img1 is None or img2 is None:
            return 0.0

        try:
            # Resize to same dimensions
            if img1.shape != img2.shape:
                img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))

            # Convert to grayscale
            gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
            gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

            # Calculate SSIM
            score = ssim(gray1, gray2)
            return max(0, min(1, score))
        except:
            return 0.0

    # ========== ALGORITHM 1: BASIC GRID ==========
    def algorithm1_basic_grid(self, pieces):
        """Simple grid reconstruction using filename coordinates"""
        if not pieces:
            return None

        # Get grid dimensions from piece coordinates
        rows = max(r for r, _ in pieces.keys()) + 1
        cols = max(c for _, c in pieces.keys()) + 1

        # Get piece dimensions from first piece
        first_pos = next(iter(pieces.keys()))
        piece_h, piece_w = pieces[first_pos].shape[:2]

        # Create output image
        output_h = rows * piece_h
        output_w = cols * piece_w
        reconstructed = np.zeros((output_h, output_w, 3), dtype=np.uint8)

        # Place pieces in grid order based on coordinates
        for (row, col), piece in pieces.items():
            y_start = row * piece_h
            y_end = (row + 1) * piece_h
            x_start = col * piece_w
            x_end = (col + 1) * piece_w
            reconstructed[y_start:y_end, x_start:x_end] = piece

        return reconstructed

    # ========== ALGORITHM 2: EDGE MATCHING ==========
    def algorithm2_edge_matching(self, pieces):
        """Edge matching for 2x2 puzzles"""
        if len(pieces) != 4:
            return self.algorithm1_basic_grid(pieces)

        # Try all 24 permutations for 2x2
        piece_list = list(pieces.values())

        best_result = None
        best_score = -1

        for perm in permutations(range(4)):
            # Create 2x2 grid
            grid = [
                [piece_list[perm[0]], piece_list[perm[1]]],
                [piece_list[perm[2]], piece_list[perm[3]]]
            ]

            # Reconstruct
            piece_h, piece_w = piece_list[0].shape[:2]
            reconstructed = np.zeros((2*piece_h, 2*piece_w, 3), dtype=np.uint8)

            for r in range(2):
                for c in range(2):
                    y_start = r * piece_h
                    y_end = (r + 1) * piece_h
                    x_start = c * piece_w
                    x_end = (c + 1) * piece_w
                    reconstructed[y_start:y_end, x_start:x_end] = grid[r][c]

            # Calculate edge continuity score
            score = self.calculate_edge_continuity_score(reconstructed)

            if score > best_score:
                best_score = score
                best_result = reconstructed

        return best_result if best_result is not None else self.algorithm1_basic_grid(pieces)

    def calculate_edge_continuity_score(self, image):
        """Calculate edge continuity score"""
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Calculate gradients
        grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        grad_mag = np.sqrt(grad_x**2 + grad_y**2)

        # Lower variance = smoother edges
        variance = np.var(grad_mag)
        if variance == 0:
            return 1.0
        return 1.0 / (1.0 + variance)

    # ========== ALGORITHM 3: COLOR HISTOGRAM ==========
    def algorithm3_color_histogram(self, pieces):
        """Color histogram matching"""
        if len(pieces) <= 1:
            return self.algorithm1_basic_grid(pieces)

        # Calculate histograms
        histograms = {}
        for pos, piece in pieces.items():
            hsv = cv2.cvtColor(piece, cv2.COLOR_BGR2HSV)
            hist = cv2.calcHist([hsv], [0, 1], None, [50, 60], [0, 180, 0, 256])
            cv2.normalize(hist, hist)
            histograms[pos] = hist.flatten()

        # Start with grid
        rows = max(r for r, _ in pieces.keys()) + 1
        cols = max(c for _, c in pieces.keys()) + 1

        grid = [[None for _ in range(cols)] for _ in range(rows)]
        for (row, col), piece in pieces.items():
            grid[row][col] = piece

        # Improve by color similarity
        improved = self.improve_by_color(grid, histograms, pieces)
        return self.reconstruct_from_position_grid(pieces, improved)

    def improve_by_color(self, grid, histograms, pieces):
        """Improve arrangement by color similarity"""
        rows = len(grid)
        cols = len(grid[0])

        best_grid = [row[:] for row in grid]
        best_score = self.calculate_color_score(grid, histograms)

        # Try swapping adjacent pieces
        for r in range(rows):
            for c in range(cols):
                # With right neighbor
                if c < cols - 1:
                    new_grid = [row[:] for row in grid]
                    new_grid[r][c], new_grid[r][c+1] = new_grid[r][c+1], new_grid[r][c]
                    new_score = self.calculate_color_score(new_grid, histograms)
                    if new_score > best_score:
                        best_score = new_score
                        best_grid = new_grid

                # With bottom neighbor
                if r < rows - 1:
                    new_grid = [row[:] for row in grid]
                    new_grid[r][c], new_grid[r+1][c] = new_grid[r+1][c], new_grid[r][c]
                    new_score = self.calculate_color_score(new_grid, histograms)
                    if new_score > best_score:
                        best_score = new_score
                        best_grid = new_grid

        return best_grid

    def calculate_color_score(self, grid, histograms):
        """Calculate color similarity score"""
        rows = len(grid)
        cols = len(grid[0])
        score = 0
        count = 0

        for r in range(rows):
            for c in range(cols):
                piece1 = grid[r][c]
                if piece1 is None:
                    continue

                # Find the position of this piece
                pos1 = None
                for pos, piece in histograms.items():
                    if np.array_equal(piece1, list(pieces.values())[0]):  # Simplified check
                        pos1 = pos
                        break

                if pos1 is None:
                    continue

                hist1 = histograms.get(pos1)
                if hist1 is None:
                    continue

                # Right neighbor
                if c < cols - 1:
                    piece2 = grid[r][c+1]
                    if piece2 is not None:
                        pos2 = None
                        for pos, piece in histograms.items():
                            if np.array_equal(piece2, list(pieces.values())[0]):  # Simplified check
                                pos2 = pos
                                break

                        if pos2 is not None:
                            hist2 = histograms.get(pos2)
                            if hist2 is not None:
                                similarity = np.dot(hist1, hist2) / (np.linalg.norm(hist1) * np.linalg.norm(hist2) + 1e-10)
                                score += similarity
                                count += 1

                # Bottom neighbor
                if r < rows - 1:
                    piece2 = grid[r+1][c]
                    if piece2 is not None:
                        pos2 = None
                        for pos, piece in histograms.items():
                            if np.array_equal(piece2, list(pieces.values())[0]):  # Simplified check
                                pos2 = pos
                                break

                        if pos2 is not None:
                            hist2 = histograms.get(pos2)
                            if hist2 is not None:
                                similarity = np.dot(hist1, hist2) / (np.linalg.norm(hist1) * np.linalg.norm(hist2) + 1e-10)
                                score += similarity
                                count += 1

        return score / count if count > 0 else 0

    def reconstruct_from_position_grid(self, pieces, grid):
        """Reconstruct from position grid"""
        rows = len(grid)
        cols = len(grid[0])

        if not pieces:
            return None

        first_piece = next(iter(pieces.values()))
        piece_h, piece_w = first_piece.shape[:2]

        reconstructed = np.zeros((rows * piece_h, cols * piece_w, 3), dtype=np.uint8)

        for r in range(rows):
            for c in range(cols):
                piece = grid[r][c]
                if piece is not None:
                    y_start = r * piece_h
                    y_end = (r + 1) * piece_h
                    x_start = c * piece_w
                    x_end = (c + 1) * piece_w
                    reconstructed[y_start:y_end, x_start:x_end] = piece

        return reconstructed

    # ========== ALGORITHM 4: FEATURE MATCHING ==========
    def algorithm4_feature_matching(self, pieces):
        """Feature matching using ORB"""
        if len(pieces) <= 1:
            return self.algorithm1_basic_grid(pieces)

        # For 2x2, try all permutations
        rows = max(r for r, _ in pieces.keys()) + 1 if pieces else 0
        cols = max(c for _, c in pieces.keys()) + 1 if pieces else 0

        if rows == 2 and cols == 2:
            return self.try_all_with_features(pieces)

        # For larger, use basic grid
        return self.algorithm1_basic_grid(pieces)

    def try_all_with_features(self, pieces):
        """Try all permutations with feature scoring"""
        if len(pieces) != 4:
            return self.algorithm1_basic_grid(pieces)

        piece_list = list(pieces.values())

        best_result = None
        best_score = -1

        for perm in permutations(range(4)):
            # Create 2x2 grid
            grid = [
                [piece_list[perm[0]], piece_list[perm[1]]],
                [piece_list[perm[2]], piece_list[perm[3]]]
            ]

            # Reconstruct
            piece_h, piece_w = piece_list[0].shape[:2]
            reconstructed = np.zeros((2*piece_h, 2*piece_w, 3), dtype=np.uint8)

            for r in range(2):
                for c in range(2):
                    y_start = r * piece_h
                    y_end = (r + 1) * piece_h
                    x_start = c * piece_w
                    x_end = (c + 1) * piece_w
                    reconstructed[y_start:y_end, x_start:x_end] = grid[r][c]

            # Calculate feature score
            score = self.calculate_feature_score(reconstructed)

            if score > best_score:
                best_score = score
                best_result = reconstructed

        return best_result if best_result is not None else self.algorithm1_basic_grid(pieces)

    def calculate_feature_score(self, image):
        """Calculate feature matching score"""
        try:
            orb = cv2.ORB_create()
            kp, des = orb.detectAndCompute(image, None)

            if des is not None and len(des) > 0:
                return len(kp) * np.mean(np.var(des, axis=0))
            return 0
        except:
            return 0

    # ========== ALGORITHM 5: TEMPLATE MATCHING ==========
    def algorithm5_template_matching(self, pieces, correct_img=None):
        """Template matching against correct image"""
        if correct_img is None:
            return self.algorithm1_basic_grid(pieces)

        if not pieces:
            return None

        rows = max(r for r, _ in pieces.keys()) + 1
        cols = max(c for _, c in pieces.keys()) + 1

        piece_h, piece_w = list(pieces.values())[0].shape[:2]

        # Resize correct image to match grid
        correct_resized = cv2.resize(correct_img, (cols * piece_w, rows * piece_h))

        # Create grid
        grid = [[None for _ in range(cols)] for _ in range(rows)]
        used_pieces = set()

        # For each position, find best matching piece
        for r in range(rows):
            for c in range(cols):
                best_piece_pos = None
                best_score = -1

                y_start = r * piece_h
                y_end = (r + 1) * piece_h
                x_start = c * piece_w
                x_end = (c + 1) * piece_w

                template = correct_resized[y_start:y_end, x_start:x_end]

                for pos, piece in pieces.items():
                    if pos in used_pieces:
                        continue

                    score = self.calculate_ssim(piece, template)

                    if score > best_score:
                        best_score = score
                        best_piece_pos = pos

                if best_piece_pos:
                    grid[r][c] = pieces[best_piece_pos]
                    used_pieces.add(best_piece_pos)

        return self.reconstruct_from_position_grid(pieces, grid)

    # ========== ALGORITHM 6: HYBRID ==========
    def algorithm6_hybrid(self, pieces, correct_img=None):
        """Hybrid approach combining multiple methods"""
        # Get results from all algorithms
        results = []

        # Algorithm 1
        result1 = self.algorithm1_basic_grid(pieces)
        if result1 is not None:
            results.append(("Basic_Grid", result1))

        # Algorithm 2 (for 2x2)
        rows = max(r for r, _ in pieces.keys()) + 1 if pieces else 0
        cols = max(c for _, c in pieces.keys()) + 1 if pieces else 0

        if rows == 2 and cols == 2:
            result2 = self.algorithm2_edge_matching(pieces)
            if result2 is not None:
                results.append(("Edge_Matching", result2))

        # Algorithm 3
        result3 = self.algorithm3_color_histogram(pieces)
        if result3 is not None:
            results.append(("Color_Histogram", result3))

        # Algorithm 4 (for 2x2)
        if rows == 2 and cols == 2:
            result4 = self.algorithm4_feature_matching(pieces)
            if result4 is not None:
                results.append(("Feature_Matching", result4))

        # Algorithm 5 (if we have correct image)
        if correct_img is not None:
            result5 = self.algorithm5_template_matching(pieces, correct_img)
            if result5 is not None:
                results.append(("Template_Matching", result5))

        # Choose best based on edge continuity
        if results:
            best_result = None
            best_score = -1
            best_name = ""

            for name, result in results:
                score = self.calculate_edge_continuity_score(result)
                if score > best_score:
                    best_score = score
                    best_result = result
                    best_name = name

            return best_result

        return self.algorithm1_basic_grid(pieces)

    # ========== MAIN SOLVING FUNCTION ==========
    def solve_puzzle_sequential(self, puzzle_type, puzzle_id):
        """Try algorithms sequentially until SSIM threshold is met"""
        print(f"  {puzzle_type} - Puzzle {puzzle_id}: ", end="")

        # Load pieces
        pieces = self.load_puzzle_pieces(puzzle_type, puzzle_id)
        if not pieces:
            print("No pieces")
            return None

        # Get expected grid size from puzzle_type
        if puzzle_type == 'puzzle_2x2':
            expected_pieces = 4
        elif puzzle_type == 'puzzle_4x4':
            expected_pieces = 16
        elif puzzle_type == 'puzzle_8x8':
            expected_pieces = 64
        else:
            expected_pieces = len(pieces)

        print(f"{len(pieces)}/{expected_pieces} pieces → ", end="")

        # If we don't have all pieces, use what we have
        if len(pieces) < expected_pieces:
            print(f"Missing {expected_pieces - len(pieces)} pieces, ", end="")

        # Load correct image
        correct_img = self.load_correct_image(puzzle_id)
        if correct_img is None:
            # print(f"No correct image, ", end="")
            pass

        # Define algorithms to try (in order)
        algorithms = [
            ("1_Basic_Grid", lambda p: self.algorithm1_basic_grid(p)),
            ("2_Edge_Matching", lambda p: self.algorithm2_edge_matching(p)),
            ("3_Color_Histogram", lambda p: self.algorithm3_color_histogram(p)),
            ("4_Feature_Matching", lambda p: self.algorithm4_feature_matching(p)),
            ("5_Template_Matching", lambda p: self.algorithm5_template_matching(p, correct_img)),
            ("6_Hybrid", lambda p: self.algorithm6_hybrid(p, correct_img))
        ]

        best_result = None
        best_ssim = 0
        best_algo = ""

        # Try algorithms in sequence
        for algo_name, algo_func in algorithms:
            try:
                # Skip template matching if no correct image
                if algo_name == "5_Template_Matching" and correct_img is None:
                    # print("T5:skip ", end="")
                    continue

                # Run algorithm
                result = algo_func(pieces.copy())
                if result is None:
                    # print(f"{algo_name}:fail ", end="")
                    continue

                # Compare with correct image
                if correct_img is not None:
                    ssim_score = self.calculate_ssim(result, correct_img)
                    # print(f"{algo_name}:{ssim_score:.3f} ", end="")

                    # Track best
                    if ssim_score > best_ssim:
                        best_ssim = ssim_score
                        best_result = result
                        best_algo = algo_name

                    # Stop if threshold met
                    if ssim_score >= self.ssim_threshold:
                        print(f"SSIM:{ssim_score:.3f} ({algo_name})")
                        self.save_result(puzzle_type, puzzle_id, result, algo_name)
                        self.stats['ssim_scores'].append(ssim_score)
                        self.stats['total_puzzles'] += 1
                        self.stats['solved_puzzles'] += 1
                        return ssim_score
                else:
                    # No correct image, just save the first working result
                    # print(f"{algo_name} ", end="")
                    self.save_result(puzzle_type, puzzle_id, result, algo_name)
                    self.stats['total_puzzles'] += 1
                    self.stats['solved_puzzles'] += 1
                    print(f"Saved ({algo_name})")
                    return 0

            except Exception as e:
                # print(f"{algo_name}:err ", end="")
                continue

        # Save best result (if we have correct image for comparison)
        if best_result is not None and correct_img is not None:
            self.save_result(puzzle_type, puzzle_id, best_result, best_algo)
            self.stats['ssim_scores'].append(best_ssim)
            self.stats['total_puzzles'] += 1
            self.stats['solved_puzzles'] += 1
            print(f"SSIM:{best_ssim:.3f} ({best_algo})")
            return best_ssim
        elif best_result is not None:
            # No correct image, just save the first working result
            self.save_result(puzzle_type, puzzle_id, best_result, "First_Working")
            self.stats['total_puzzles'] += 1
            self.stats['solved_puzzles'] += 1
            print(f"Saved (First_Working)")
            return 0
        else:
            # Fallback - try basic grid one more time
            try:
                result = self.algorithm1_basic_grid(pieces)
                if result is not None:
                    self.save_result(puzzle_type, puzzle_id, result, "Fallback")
                    self.stats['total_puzzles'] += 1
                    self.stats['solved_puzzles'] += 1
                    print(f"Saved (Fallback)")
                    return 0
            except:
                pass

        print("Failed")
        return None

    def save_result(self, puzzle_type, puzzle_id, image, algorithm):
        """Save result to output folder"""
        output_file = os.path.join(self.output_path, puzzle_type, f"{puzzle_id}.jpg")
        cv2.imwrite(output_file, image)

        # Update statistics
        self.stats['algorithm_usage'][algorithm] = self.stats['algorithm_usage'].get(algorithm, 0) + 1

    def debug_folder_structure(self):
        """Debug the actual folder structure"""
        print("\n🔍 DEBUG: Checking folder structure...")
        for puzzle_type in ['puzzle_2x2', 'puzzle_4x4', 'puzzle_8x8']:
            folder = os.path.join(self.dataset_path, puzzle_type)
            if os.path.exists(folder):
                print(f"\n{puzzle_type}:")
                files = os.listdir(folder)
                print(f"  Total files: {len(files)}")

                # Check first 3 puzzles
                for puzzle_id in range(3):
                    puzzle_files = []
                    for f in files:
                        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                            base_name = os.path.splitext(f)[0]
                            parts = base_name.split('_')
                            if parts and parts[0] == str(puzzle_id):
                                puzzle_files.append(f)

                    if puzzle_files:
                        print(f"  Puzzle {puzzle_id}: {len(puzzle_files)} files")
                        print(f"    Sample: {sorted(puzzle_files)[:3]}")

    def process_all_puzzles(self):
        """Process all puzzles from 0 to 109"""
        print("🚀 Starting Sequential Puzzle Solver")
        print(f"Dataset: {self.dataset_path}")
        print(f"Correct: {self.correct_path}")
        print(f"Output: {self.output_path}")
        print(f"SSIM Threshold: {self.ssim_threshold}")
        print("="*70)

        puzzle_types = ['puzzle_2x2', 'puzzle_4x4', 'puzzle_8x8']
        puzzle_ids = list(range(110))  # 0 to 109

        ssim_scores = []

        for puzzle_type in puzzle_types:
            print(f"\n{puzzle_type}:")
            print("-"*40)

            type_scores = []

            for puzzle_id in puzzle_ids:
                score = self.solve_puzzle_sequential(puzzle_type, puzzle_id)
                if score is not None and score > 0:
                    ssim_scores.append(score)
                    type_scores.append(score)

            if type_scores:
                print(f"\n{puzzle_type} summary:")
                print(f"  Processed: {len([id for id in puzzle_ids if self.load_puzzle_pieces(puzzle_type, id)])} puzzles")
                print(f"  Average SSIM: {np.mean(type_scores):.3f}")
                print(f"  Best SSIM: {np.max(type_scores):.3f}" if type_scores else "  No SSIM scores")

        # Final summary
        print(f"\n{'='*70}")
        print("FINAL SUMMARY")
        print(f"{'='*70}")
        print(f"Total puzzles processed: {self.stats['solved_puzzles']}")

        if self.stats['ssim_scores']:
            scores = self.stats['ssim_scores']
            print(f"Average SSIM: {np.mean(scores):.4f}")
            print(f"Best SSIM: {np.max(scores):.4f}")
            print(f"Worst SSIM: {np.min(scores):.4f}")

            # Above threshold
            above = sum(1 for s in scores if s >= self.ssim_threshold)
            print(f"Above threshold ({self.ssim_threshold}): {above}/{len(scores)} ({above/len(scores)*100:.1f}%)")

        print("\nAlgorithm usage:")
        print("-"*40)
        total_usage = sum(self.stats['algorithm_usage'].values())
        for algo, count in sorted(self.stats['algorithm_usage'].items(), key=lambda x: x[1], reverse=True):
            percentage = count / total_usage * 100 if total_usage > 0 else 0
            print(f"  {algo:<20} {count:>4} ({percentage:>5.1f}%)")

        print(f"\nResults saved to: {self.output_path}")

# Quick test first
def test_loading():
    print("🔍 Testing file loading...")
    DATASET_PATH = "/content/Task3_output"

    for puzzle_type in ['puzzle_2x2', 'puzzle_4x4', 'puzzle_8x8']:
        folder = os.path.join(DATASET_PATH, puzzle_type)
        if os.path.exists(folder):
            # Try to load puzzle 0
            files = [f for f in os.listdir(folder) if f.lower().endswith('.jpg')]
            puzzle0_files = []
            for f in files:
                base_name = os.path.splitext(f)[0]
                parts = base_name.split('_')
                if parts and parts[0] == '0':
                    puzzle0_files.append(f)

            print(f"{puzzle_type}: {len(puzzle0_files)} files for puzzle 0")
            if puzzle0_files:
                print(f"  Sample: {sorted(puzzle0_files)[:3]}")

# Run it
if __name__ == "__main__":
    # Test loading first
    test_loading()

    print("\n" + "="*70)

    # Create solver
    solver = CompletePuzzleSolver(
        dataset_path="/content/Task3_output",
        correct_path="/content/Dataset/Dataset/correct",
        output_path="/content/complete_output",
        ssim_threshold=0.6
    )

    # Debug folder structure
    solver.debug_folder_structure()

    print("\n" + "="*70)

    # Run solver
    solver.process_all_puzzles()

🔍 Testing file loading...
puzzle_2x2: 4 files for puzzle 0
  Sample: ['0_r0_c0.jpg', '0_r0_c1.jpg', '0_r1_c0.jpg']
puzzle_4x4: 16 files for puzzle 0
  Sample: ['0_r0_c0.jpg', '0_r0_c1.jpg', '0_r0_c2.jpg']
puzzle_8x8: 64 files for puzzle 0
  Sample: ['0_r0_c0.jpg', '0_r0_c1.jpg', '0_r0_c2.jpg']


🔍 DEBUG: Checking folder structure...

puzzle_2x2:
  Total files: 440
  Puzzle 0: 4 files
    Sample: ['0_r0_c0.jpg', '0_r0_c1.jpg', '0_r1_c0.jpg']
  Puzzle 1: 4 files
    Sample: ['1_r0_c0.jpg', '1_r0_c1.jpg', '1_r1_c0.jpg']
  Puzzle 2: 4 files
    Sample: ['2_r0_c0.jpg', '2_r0_c1.jpg', '2_r1_c0.jpg']

puzzle_4x4:
  Total files: 1760
  Puzzle 0: 16 files
    Sample: ['0_r0_c0.jpg', '0_r0_c1.jpg', '0_r0_c2.jpg']
  Puzzle 1: 16 files
    Sample: ['1_r0_c0.jpg', '1_r0_c1.jpg', '1_r0_c2.jpg']
  Puzzle 2: 16 files
    Sample: ['2_r0_c0.jpg', '2_r0_c1.jpg', '2_r0_c2.jpg']

puzzle_8x8:
  Total files: 7040
  Puzzle 0: 64 files
    Sample: ['0_r0_c0.jpg', '0_r0_c1.jpg', '0_r0_c2.jpg']
  Puzzle 1: 64 fi